In [1]:
# Local C++/Python mode imports
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "build" / "Debug"))

from vector_db import VectorSearchDB # Use python-wrapped C++ implementation.

API_URL = "http://localhost:8000"
API_BATCH_SIZE = 256
API_DATASET_LIMIT = 20  # Validate with a small subset first; set to None for all loaded rows.

In [3]:
# Load the embedding model.
model = SentenceTransformer("all-MiniLM-L6-v2")

dimension = model.get_embedding_dimension()
dimension

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384

In [4]:
# Load dataset.
dataset = load_dataset("fancyzhx/ag_news", split="train")

texts = dataset["text"][:20_000]
labels = dataset["label"][:20_000]

len(texts)

20000

In [5]:
# Create embeddings and record embedding time.
start = time.perf_counter()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=False,
)

embedding_time = time.perf_counter() - start

embeddings.shape, embedding_time

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

((20000, 384), 127.70920879999994)

In [6]:
# Insert vectors into the C++ vector database and time the process.
db = VectorSearchDB(dimension)

start = time.perf_counter()

for vector_id, embedding in enumerate(embeddings):
    db.insert(vector_id, embedding.tolist())

insertion_time = time.perf_counter() - start

print(f"Inserted {len(embeddings):,} vectors")
print(f"Insertion time: {insertion_time:.3f} seconds")
print(f"Average insertion time: {insertion_time / len(embeddings) * 1_000_000:.2f} µs")

Inserted 20,000 vectors
Insertion time: 0.763 seconds
Average insertion time: 38.15 µs


In [11]:
query_text = "A new technology company announced a major software product."

query_embedding = model.encode(
    query_text,
    normalize_embeddings=False,
).tolist()

In [12]:
# Local C++/Python query mode.
start = time.perf_counter()

results = db.search(query_embedding, 5)

query_time = time.perf_counter() - start

print(f"Query time: {query_time * 1_000:.3f} ms")

for result in results:
    print(f"{result['id']}: {result['score']:.4f}")
    print(texts[result['id']])
    print()

Query time: 19.042 ms
7714: 0.5543
Cisco, Microsoft step up small-business push The two tech giants bring corporate-level customer management software to mom-and-pop shops.

3003: 0.5476
A brighter Outlook? Software startup hopes there's big money in little improvements to Microsoft products.

8706: 0.5309
Cisco and Microsoft target smaller firms Cisco Systems and Microsoft #39;s partnership in the small- and midsized-business market is bearing fruit. On Monday, Cisco announced a new software product that will link its voice over Internet protocol products 

13515: 0.5290
New version of Windows planned for 2006 Microsoft Corp., world #39;s largest software maker, announced yesterday that it will release the next version of Windows operating system, code-named Longhorn, in the second half of 2006.

11805: 0.5285
Intel and Linksys deliver  quot;simpler quot; Wi-Fi software Intel Corp. and Linksys, a division of Cisco Systems Inc., yesterday announced the delivery of software that the com

## HTTP API mode

The following cells use the Dockerized VectorDB service at `http://localhost:8000`. They insert embeddings in batches through `POST /vectors`, then search through `POST /search`. This is separate from the local C++/Python mode above. Start the service before running these cells:

```powershell
docker run --rm --name vector-db-api -p 8000:8000 vector-db
```

The API validation starts with `API_DATASET_LIMIT = 20`; set it to `None` only after the small-subset check passes.

In [13]:
# HTTP API helpers.
def _api_error(response, operation):
    try:
        detail = response.json().get("detail", response.text)
    except ValueError:
        detail = response.text
    return RuntimeError(f"{operation} failed with HTTP {response.status_code}: {detail}")


def api_health():
    try:
        response = requests.get(f"{API_URL}/health", timeout=10)
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not reach {API_URL}. Start the Dockerized VectorDB service first."
        ) from exc
    if not response.ok:
        raise _api_error(response, "Health check")
    return response.json()


def insert_vectors_api(embeddings, batch_size=API_BATCH_SIZE, limit=API_DATASET_LIMIT):
    health = api_health()
    expected_dimension = health.get("dimension")
    if embeddings.ndim != 2 or embeddings.shape[1] != expected_dimension:
        raise ValueError(
            f"Embedding dimension mismatch: API expects {expected_dimension}, "
            f"received {embeddings.shape[1] if embeddings.ndim == 2 else 'non-matrix data'}."
        )

    vector_count = len(embeddings) if limit is None else min(limit, len(embeddings))
    if vector_count == 0:
        raise ValueError("No embeddings were selected for API insertion.")

    inserted = 0
    start_time = time.perf_counter()
    for start in range(0, vector_count, batch_size):
        batch = [
            {"id": vector_id, "values": embeddings[vector_id].tolist()}
            for vector_id in range(start, min(start + batch_size, vector_count))
        ]
        try:
            response = requests.post(
                f"{API_URL}/vectors",
                json={"vectors": batch},
                timeout=60,
            )
        except requests.RequestException as exc:
            raise RuntimeError(f"Vector batch starting at {start} could not be sent.") from exc
        if not response.ok:
            raise _api_error(response, f"Vector batch starting at {start}")
        inserted += response.json().get("inserted", 0)

    elapsed = time.perf_counter() - start_time
    print(f"Inserted {inserted:,} vectors in {(elapsed):.3f} seconds")
    return inserted


def search_api(query_embedding, top_k=5):
    health = api_health()
    expected_dimension = health.get("dimension")
    if len(query_embedding) != expected_dimension:
        raise ValueError(
            f"Query dimension mismatch: API expects {expected_dimension}, "
            f"received {len(query_embedding)}."
        )

    try:
        response = requests.post(
            f"{API_URL}/search",
            json={"query": [float(value) for value in query_embedding], "top_k": top_k},
            timeout=60,
        )
    except requests.RequestException as exc:
        raise RuntimeError("The API search request could not be sent.") from exc
    if not response.ok:
        raise _api_error(response, "Search")

    results = response.json().get("results", [])
    if not results:
        print("The API returned no search results.")
    return results


def display_api_results(results, texts, labels=None):
    if not results:
        return
    for result in results:
        vector_id = result["id"]
        print(f"{vector_id}: cosine score={result['score']:.4f}")
        if vector_id < 0 or vector_id >= len(texts):
            print("No matching AG News text is available for this result ID.")
            continue
        print(f"Label: {labels[vector_id] if labels is not None else 'n/a'}")
        print(texts[vector_id])
        print()

In [14]:
# Validate the HTTP API with a small AG News subset.
api_health()
inserted_count = insert_vectors_api(
    embeddings,
    batch_size=API_BATCH_SIZE,
    limit=API_DATASET_LIMIT,
)

api_results = search_api(query_embedding, top_k=5)
display_api_results(api_results, texts, labels)

print(f"API stats after insertion: {requests.get(f'{API_URL}/stats', timeout=10).json()}")

Inserted 20 vectors in 0.021 seconds
1: cosine score=0.2012
Label: 2
Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.

5: cosine score=0.1731
Label: 2
Stocks End Up, But Near Year Lows (Reuters) Reuters - Stocks ended slightly higher on Friday\but stayed near lows for the year as oil prices surged past  #36;46\a barrel, offsetting a positive outlook from computer maker\Dell Inc. (DELL.O)

13: cosine score=0.1637
Label: 2
Google IPO Auction Off to Rocky Start  WASHINGTON/NEW YORK (Reuters) - The auction for Google  Inc.'s highly anticipated initial public offering got off to a  rocky start on Friday after the Web search company sidestepped  a bullet from U.S. securities regulators.

18: cosine score=0.1527
Label: 2
US trade deficit swells in June The US trade deficit has e